# Feature preparation: missing values, categories, scaling

A recruitment agency has a table of 12 past candidates with blank salaries, city names typed in many ways, and salaries in three currencies. You will prepare every column so a nearest-neighbours model can use it, then fix a weekly run that prepares new candidates the wrong way.

About 10 minutes. Run the cells from top to bottom.

## Step 1: install the pandas version the lesson used

The lesson's outputs come from pandas 3.1.1 and scikit-learn 1.9.1. This cell installs pandas 3.1.1. The numbers in this notebook do not depend on the scikit-learn version.

In [1]:
%pip install -q pandas==3.1.1

Note: you may need to restart the kernel to use updated packages.


## Step 2: the agency's two tables

This cell writes `candidates.csv` (the 12 past candidates) and `new_candidates.csv` (this week's 3), reads the past table and prints it. The column `placed` (1 or 0) is what the agency wants to predict, so it is not a feature.

**Look for:** `pandas version: 3.1.1` (if not, use Runtime > Restart session and run this cell again), `NaN` in two salary cells (C-103 and C-108), cities typed in different ways, and three currencies.

In [2]:
import numpy as np
import pandas as pd

CANDIDATES_CSV = """candidate_id,city,years_exp,salary,currency,placed
C-101,Berlin,6,62000,EUR,1
C-102, berlin,2,41000,EUR,0
C-103,Munich,9,,EUR,1
C-104,BERLIN,4,58000,USD,1
C-105,Hamburg,3,230000,PLN,0
C-106,Munich,12,150000,EUR,1
C-107,Leipzig,1,36000,EUR,0
C-108,berlin ,7,,EUR,1
C-109,Hamburg,5,56000,EUR,0
C-110,Munich,8,280000,PLN,1
C-111,Leipzig,3,44000,EUR,0
C-112,Berlin,5,52000,EUR,1
"""
NEW_CSV = """candidate_id,city,years_exp,salary,currency
N-201,Berlin,5,58000,EUR
N-202,munich,10,,EUR
N-203,Berlin,2,45000,USD
"""
with open("candidates.csv", "w") as f:
    f.write(CANDIDATES_CSV)
with open("new_candidates.csv", "w") as f:
    f.write(NEW_CSV)

EUR_PER = {"EUR": 1.0, "USD": 0.9, "PLN": 0.23}  # example rates for the lesson, not market rates

candidates = pd.read_csv("candidates.csv")
print("pandas version:", pd.__version__)
print(candidates.to_string(index=False))
print("blank counts:", candidates.isna().sum().to_dict())

pandas version: 3.1.1
candidate_id    city  years_exp   salary currency  placed
       C-101  Berlin          6  62000.0      EUR       1
       C-102  berlin          2  41000.0      EUR       0
       C-103  Munich          9      NaN      EUR       1
       C-104  BERLIN          4  58000.0      USD       1
       C-105 Hamburg          3 230000.0      PLN       0
       C-106  Munich         12 150000.0      EUR       1
       C-107 Leipzig          1  36000.0      EUR       0
       C-108 berlin           7      NaN      EUR       1
       C-109 Hamburg          5  56000.0      EUR       0
       C-110  Munich          8 280000.0      PLN       1
       C-111 Leipzig          3  44000.0      EUR       0
       C-112  Berlin          5  52000.0      EUR       1
blank counts: {'candidate_id': 0, 'city': 0, 'years_exp': 0, 'salary': 2, 'currency': 0, 'placed': 0}


## Step 3: one spelling per city

`.str.strip()` removes spaces at both ends and `.str.title()` makes one capital letter. The quotes show the spaces.

**Look for:** 7 different values as typed, and 4 cities after cleaning.

In [3]:
typed = candidates["city"].unique()
clean = sorted(candidates["city"].str.strip().str.title().unique())
print("city values as typed:", len(typed), [repr(v) for v in typed])
print("city after strip+title:", len(clean), clean)

city values as typed: 7 ["'Berlin'", "' berlin'", "'Munich'", "'BERLIN'", "'Hamburg'", "'Leipzig'", "'berlin '"]
city after strip+title: 4 ['Berlin', 'Hamburg', 'Leipzig', 'Munich']


## Step 4: scale first, fix later? (the wrong order)

A colleague scales the salary column as it is: subtract the mean, divide by the spread. Values near 0 are typical; big ones are far above the rest.

**Look for:** C-105 gets `1.46`, the second-highest ask after C-110. Both are zloty. Scaling changed the range, not the meaning.

In [4]:
raw = candidates["salary"]
z_raw = (raw - raw.mean()) / raw.std()
for cid, cur, s, z in zip(candidates["candidate_id"], candidates["currency"], raw, z_raw):
    print(f"{cid} {cur} salary={s} scaled={'' if np.isnan(z) else round(z, 2)}")

C-101 EUR salary=62000.0 scaled=-0.44
C-102 EUR salary=41000.0 scaled=-0.68
C-103 EUR salary=nan scaled=
C-104 USD salary=58000.0 scaled=-0.49
C-105 PLN salary=230000.0 scaled=1.46
C-106 EUR salary=150000.0 scaled=0.56
C-107 EUR salary=36000.0 scaled=-0.74
C-108 EUR salary=nan scaled=
C-109 EUR salary=56000.0 scaled=-0.51
C-110 PLN salary=280000.0 scaled=2.03
C-111 EUR salary=44000.0 scaled=-0.65
C-112 EUR salary=52000.0 scaled=-0.55


## Step 5: convert to euros, then pick a fill value

Each currency becomes its rate, and multiplying gives euros. A blank stays blank. Then compare the mean and the median as a fill value for the two blanks.

**Look for:** C-105 is now 52,900 EUR, an ordinary ask. Mean `61050.0`, median `52550.0`. Without C-106 (150,000 EUR) the mean drops to `51166.67`: one high ask moves the mean, not the median.

In [5]:
salary_eur = candidates["salary"] * candidates["currency"].map(EUR_PER)
for cid, cur, s, e in zip(candidates["candidate_id"], candidates["currency"], candidates["salary"], salary_eur):
    print(f"{cid} {cur} {s} -> {e}")
print("mean:", round(salary_eur.mean(), 2), "median:", salary_eur.median())
print("mean without C-106:", round(salary_eur[candidates["candidate_id"] != "C-106"].mean(), 2))

C-101 EUR 62000.0 -> 62000.0
C-102 EUR 41000.0 -> 41000.0
C-103 EUR nan -> nan
C-104 USD 58000.0 -> 52200.0
C-105 PLN 230000.0 -> 52900.0
C-106 EUR 150000.0 -> 150000.0
C-107 EUR 36000.0 -> 36000.0
C-108 EUR nan -> nan
C-109 EUR 56000.0 -> 56000.0
C-110 PLN 280000.0 -> 64400.0
C-111 EUR 44000.0 -> 44000.0
C-112 EUR 52000.0 -> 52000.0
mean: 61050.0 median: 52550.0
mean without C-106: 51166.67


## Step 6: who is most like the new candidate?

New candidate N-201: Berlin, 5 years, asks 58,000 EUR. This cell measures the distance to every past candidate twice: on raw numbers, and after scaling years and salary. City is one-hot (one 0/1 column per city).

**Look for:** unscaled, the nearest 3 are `C-109, C-101, C-105`, chosen almost only by salary. Scaled, they are `C-112, C-101, C-104`: same city, similar years. The `raw` column of C-109 is 2000.0, but its `scaled` distance is 1.416.

In [6]:
table = candidates.copy()
table["city"] = table["city"].str.strip().str.title()
table["salary_eur"] = salary_eur.fillna(salary_eur.median())
X = pd.get_dummies(table[["city", "years_exp", "salary_eur"]], columns=["city"], dtype=int)

def distances(new_row, scale):
    new = pd.DataFrame([new_row])
    newX = pd.get_dummies(new, columns=["city"], dtype=int).reindex(columns=X.columns, fill_value=0)
    A, B = X.astype(float).copy(), newX.astype(float).copy()
    if scale:
        mean, std = X[["years_exp", "salary_eur"]].mean(), X[["years_exp", "salary_eur"]].std()
        for c in ["years_exp", "salary_eur"]:
            A[c] = (X[c] - mean[c]) / std[c]
            B[c] = (newX[c] - mean[c]) / std[c]
    return np.sqrt(((A.values - B.values) ** 2).sum(axis=1))

N201 = {"city": "Berlin", "years_exp": 5, "salary_eur": 58000.0}
raw_d, sc_d = distances(N201, scale=False), distances(N201, scale=True)
report = table[["candidate_id", "city", "years_exp", "salary_eur"]].assign(raw=raw_d.round(2), scaled=sc_d.round(3))
print(report.to_string(index=False))
print("nearest 3 raw:   ", list(table["candidate_id"][np.argsort(raw_d, kind="stable")[:3]]))
print("nearest 3 scaled:", list(table["candidate_id"][np.argsort(sc_d, kind="stable")[:3]]))

candidate_id    city  years_exp  salary_eur     raw  scaled
       C-101  Berlin          6     62000.0  4000.0   0.343
       C-102  Berlin          2     41000.0 17000.0   1.106
       C-103  Munich          9     52550.0  5450.0   1.903
       C-104  Berlin          4     52200.0  5800.0   0.371
       C-105 Hamburg          3     52900.0  5100.0   1.558
       C-106  Munich         12    150000.0 92000.0   4.067
       C-107 Leipzig          1     36000.0 22000.0   2.035
       C-108  Berlin          7     52550.0  5450.0   0.656
       C-109 Hamburg          5     56000.0  2000.0   1.416
       C-110  Munich          8     64400.0  6400.0   1.714
       C-111 Leipzig          3     44000.0 14000.0   1.619
       C-112  Berlin          5     52000.0  6000.0   0.203
nearest 3 raw:    ['C-109', 'C-101', 'C-105']
nearest 3 scaled: ['C-112', 'C-101', 'C-104']


## Step 7: every step in one function

`prepare()` is the code from the lesson, line for line: drop the id, clean the city, convert to euros, flag the blanks **before** filling them, fill with the median, drop the raw salary and currency, one-hot the city, scale years and salary.

**Look for:** `12 rows, 7 columns`, `NaN left: 0`, and the row of C-108 (index 7): `salary_blank` is 1 and its `salary_eur` is `-0.240`.

In [7]:
def prepare(candidates):
    table = candidates.drop(columns=["candidate_id"])
    table["city"] = table["city"].str.strip().str.title()
    table["salary_eur"] = table["salary"] * table["currency"].map(EUR_PER)
    table["salary_blank"] = table["salary_eur"].isna().astype(int)
    table["salary_eur"] = table["salary_eur"].fillna(table["salary_eur"].median())
    table = table.drop(columns=["salary", "currency"])
    table = pd.get_dummies(table, columns=["city"], dtype=int)
    for col in ["years_exp", "salary_eur"]:
        table[col] = (table[col] - table[col].mean()) / table[col].std()
    return table

past = pd.read_csv("candidates.csv")
placed = past.pop("placed")
train_X = prepare(past)
print(f"{train_X.shape[0]} rows, {train_X.shape[1]} columns")
print("NaN left:", int(train_X.isna().sum().sum()))
print(train_X.round(3).to_string())

12 rows, 7 columns
NaN left: 0
    years_exp  salary_eur  salary_blank  city_Berlin  city_Hamburg  city_Leipzig  city_Munich
0       0.184       0.080             0            1             0             0            0
1      -1.076      -0.630             0            1             0             0            0
2       1.128      -0.240             1            0             0             0            1
3      -0.446      -0.251             0            1             0             0            0
4      -0.761      -0.228             0            0             1             0            0
5       2.073       3.056             0            0             0             0            1
6      -1.391      -0.799             0            0             0             1            0
7       0.499      -0.240             1            1             0             0            0
8      -0.131      -0.123             0            0             1             0            0
9       0.814       0.161    

## Step 8: Monday's run crashes (broken on purpose)

The agency fits a 3-nearest-neighbours model on the 12 prepared past candidates. On Monday it runs the same `prepare()` on the 3 new candidates alone and asks for a prediction. This cell is meant to fail: it catches the error and prints it.

**Look for:** the new table has only 5 columns, and a `ValueError` saying `city_Hamburg` and `city_Leipzig` are missing. The new rows only contain Berlin and Munich, so `get_dummies` made columns for those two only.

In [8]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3).fit(train_X, placed)
new = pd.read_csv("new_candidates.csv")
new_X = prepare(new)
print("training columns:", list(train_X.columns))
print("new columns:     ", list(new_X.columns))
try:
    print(model.predict(new_X))
except ValueError as e:
    print("ValueError:", e)

training columns: ['years_exp', 'salary_eur', 'salary_blank', 'city_Berlin', 'city_Hamburg', 'city_Leipzig', 'city_Munich']
new columns:      ['years_exp', 'salary_eur', 'salary_blank', 'city_Berlin', 'city_Munich']
ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- city_Hamburg
- city_Leipzig



## Step 9: only the columns fixed

A quick patch: reindex the new table to the training columns and fill the missing ones with 0. The crash is gone. But the median, the mean and the spread in `prepare()` were still computed from the 3 new rows.

**Look for:** N-201's `salary_eur` is `1.0`, "top of the group", only because 58,000 is the highest of three asks. The predictions run: `[1, 1, 1]`.

In [9]:
patched = new_X.reindex(columns=train_X.columns, fill_value=0)
print(patched.round(3).to_string(index=False))
print("predict:", model.predict(patched).tolist())
print("3 nearest past candidates:", [[past["candidate_id"][j] for j in row] for row in model.kneighbors(patched, return_distance=False)])

 years_exp  salary_eur  salary_blank  city_Berlin  city_Hamburg  city_Leipzig  city_Munich
    -0.165         1.0             0            1             0             0            0
     1.072         0.0             1            0             0             0            1
    -0.907        -1.0             0            1             0             0            0
predict: [1, 1, 1]
3 nearest past candidates: [['C-101', 'C-112', 'C-104'], ['C-103', 'C-110', 'C-108'], ['C-102', 'C-104', 'C-112']]


## Step 10: the real fix: learn once from the past, apply to new rows

`learn()` computes every choice from the 12 past rows once: the fill median, the means and spreads, the column list. `apply()` only uses them. This is what scikit-learn's `fit` (learn) and `transform` (apply) do.

**Look for:** `learned median: 52550.0`, `same as prepare() on the past rows: True`, and N-201's `salary_eur` is now `-0.055`, a typical ask. N-202's blank got the past median, so its `salary_eur` is `-0.240`, the same as C-108 in step 7.

In [10]:
def learn(rows):
    salary_eur = rows["salary"] * rows["currency"].map(EUR_PER)
    filled = salary_eur.fillna(salary_eur.median())
    years = rows["years_exp"]
    return {"median": salary_eur.median(),
            "mean": {"years_exp": years.mean(), "salary_eur": filled.mean()},
            "std": {"years_exp": years.std(), "salary_eur": filled.std()},
            "columns": list(train_X.columns)}


def apply(rows, learned):
    table = rows.drop(columns=["candidate_id"])
    table["city"] = table["city"].str.strip().str.title()
    table["salary_eur"] = table["salary"] * table["currency"].map(EUR_PER)
    table["salary_blank"] = table["salary_eur"].isna().astype(int)
    table["salary_eur"] = table["salary_eur"].fillna(learned["median"])
    table = table.drop(columns=["salary", "currency"])
    table = pd.get_dummies(table, columns=["city"], dtype=int)
    table = table.reindex(columns=learned["columns"], fill_value=0)
    for col in ["years_exp", "salary_eur"]:
        table[col] = (table[col] - learned["mean"][col]) / learned["std"][col]
    return table


learned = learn(past)
print("learned median:", float(learned["median"]))
print("learned mean:", {k: round(float(v), 4) for k, v in learned["mean"].items()}, "std:", {k: round(float(v), 4) for k, v in learned["std"].items()})
fixed = apply(new, learned)
print("same as prepare() on the past rows:", bool((apply(past, learned) - train_X).abs().max().max() < 1e-9))
print(fixed.round(3).to_string(index=False))
print("predict:", model.predict(fixed).tolist())
print("3 nearest past candidates:", [[past["candidate_id"][j] for j in row] for row in model.kneighbors(fixed, return_distance=False)])

learned median: 52550.0
learned mean: {'years_exp': 5.4167, 'salary_eur': 59633.3333} std: {'years_exp': 3.1754, 'salary_eur': 29568.1058}
same as prepare() on the past rows: True
 years_exp  salary_eur  salary_blank  city_Berlin  city_Hamburg  city_Leipzig  city_Munich
    -0.131      -0.055             0            1             0             0            0
     1.443      -0.240             1            0             0             0            1
    -1.076      -0.647             0            1             0             0            0
predict: [1, 1, 1]
3 nearest past candidates: [['C-112', 'C-101', 'C-104'], ['C-103', 'C-110', 'C-108'], ['C-102', 'C-104', 'C-112']]


## Now change it

Three small experiments. Each cell has one line marked `# <- change this`. Edit it, run the cell, and compare with the output above.

### Experiment 1: fill the blanks with the mean

Change `"median"` to `"mean"`. The two blank salaries (C-103 and C-108) get the mean, which C-106's 150,000 EUR pulls up. The fill value should change from `52550.0` to `61050.0`, and their scaled `salary_eur` from `-0.240` to `0.000`: filling with the mean puts them exactly at the average, which C-106 pulled up, so they now look like higher asks than most real candidates.

In [11]:
FILL = "median"  # <- change this to "mean"

salary_eur = past["salary"] * past["currency"].map(EUR_PER)
fill_value = getattr(salary_eur, FILL)()
filled = salary_eur.fillna(fill_value)
scaled = (filled - filled.mean()) / filled.std()
print(f"fill value ({FILL}): {round(fill_value, 2)}")
for cid in ["C-103", "C-108"]:
    i = past.index[past["candidate_id"] == cid][0]
    print(f"{cid} salary_eur={filled[i]} scaled={scaled[i]:.3f}")

fill value (median): 52550.0
C-103 salary_eur=52550.0 scaled=-0.240
C-108 salary_eur=52550.0 scaled=-0.240


### Experiment 2: a new candidate with more experience

Change N-201's `years_exp` from 5 to 10. Unscaled, the nearest 3 should hardly change, because 5 years is tiny next to salary gaps in the thousands. Scaled, the list should become `C-108, C-101, C-103`: C-108 (7 years) and C-103 (9 years) move in, because now one typical spread of years counts as much as one typical spread of salary.

In [12]:
other = {"city": "Berlin", "years_exp": 5, "salary_eur": 58000.0}  # <- change this: years_exp 5 -> 10

for scale in [False, True]:
    d = distances(other, scale=scale)
    print("scaled  " if scale else "unscaled", list(table["candidate_id"][np.argsort(d, kind="stable")[:3]]))

unscaled ['C-109', 'C-101', 'C-105']
scaled   ['C-112', 'C-101', 'C-104']


### Experiment 3: a city the model has never seen

Change N-203's city from `"Berlin"` to `"Dresden"`. With the learned column list, `apply()` gives Dresden 0 in every city column instead of crashing, and the model still predicts. N-203's prediction should change from 1 to 0: without Berlin, its nearest neighbours are now chosen by years and salary alone. This is what scikit-learn's `OneHotEncoder(handle_unknown="ignore")` does.

In [13]:
week = pd.read_csv("new_candidates.csv")
week.loc[2, "city"] = "Berlin"  # <- change this to "Dresden"

week_X = apply(week, learned)
print(week_X.round(3).to_string(index=False))
print("predict:", model.predict(week_X).tolist())

 years_exp  salary_eur  salary_blank  city_Berlin  city_Hamburg  city_Leipzig  city_Munich
    -0.131      -0.055             0            1             0             0            0
     1.443      -0.240             1            0             0             0            1
    -1.076      -0.647             0            1             0             0            0
predict: [1, 1, 1]


## What you should see

- The raw city column prints 7 different values; after `.str.strip().str.title()` there are 4.
- The salaries in euros have a mean of 61,050 and a median of 52,550.
- `prepare()` on the past candidates returns 12 rows and 7 columns, with no NaN left.
- `predict` on the separately prepared new rows raises a ValueError naming city_Hamburg and city_Leipzig.
- With the median, means and spreads learned from the past rows, N-201's salary becomes -0.055.